# Multi-Hop RAG Diagnosis: Rigorous Evaluation, Controls, Ablations & Statistics
### Complete 500-Question Pipeline on HotpotQA (Local T4 GPU with Qwen-2.5-3B-Instruct)

### Experimental Highlights:
- **Baselines ($S0$–$S9$):** Closed-Book ($S0$), BM25 ($S1$), Dense BGE ($S2$), Hybrid RRF ($S3$), Hybrid + Reranker ($S5$),
  Fair Pool Control ($S5_{50}$), Context Budget Control ($K=10$), True Iterative Query Augmentation ($S7$),
  Full Multi-Hop Pipeline ($S8$), and Oracle ($S9$).
- **Ablation Study:** Minus Reranker, Minus BM25 (Dense only), Minus Dense (BM25 only).
- **Statistical Testing:** 10,000 paired bootstrap resamples with Holm-Bonferroni correction.
- **Multi-Retriever Depth Curve:** BM25 vs Dense vs Hybrid at depth 5, 10, 25, 50.
- **Recovery Stages Diagnostic Figure:** Tracks Hop 1 vs Hop 2 evidence discovery.

## Step 1: Install Dependencies
Run this cell in Colab.

In [ ]:
!pip install -q numpy pandas rank_bm25 sentence-transformers faiss-cpu tqdm torch matplotlib seaborn datasets

## Step 2: Configuration (Local T4 GPU - No API Key Needed)

In [ ]:
import os, sys, json, random, re, string, collections, urllib.request, pickle, time, hashlib
import numpy as np, pandas as pd
from tqdm.auto import tqdm

# ================= USER CONFIGURATION ================
SEED = 42
N_CORPUS = 2000          # questions whose paragraphs form the pooled corpus
N_EVAL = 500             # evaluation questions (500 questions for full paper)
TOP_K = 5                # context passages fed to generator
USE_GOOGLE_DRIVE = True  # saves results permanently to Google Drive

# Uses free local Qwen-2.5-3B-Instruct model (runs directly on Colab T4 GPU!)
LLM_BACKEND = 'qwen_3b'
# =====================================================

random.seed(SEED); np.random.seed(SEED)

try:
    from google.colab import drive
    if USE_GOOGLE_DRIVE:
        drive.mount('/content/drive')
        OUT = '/content/drive/MyDrive/rag_paper'
    else:
        OUT = '/content/rag_paper'
except ImportError:
    OUT = os.path.abspath('./rag_paper')

DATA_PATH = os.path.join(OUT, 'hotpot_dev.json')
CACHE_DIR = os.path.join(OUT, 'cache')
os.makedirs(OUT, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Persistence directory: {OUT}')

## Step 3: Dataset Loading & Pooled Corpus Construction

In [ ]:
def load_hotpot_dev():
    try:
        from datasets import load_dataset
        ds = load_dataset('hotpotqa/hotpot_qa', 'distractor', split='validation')
        out = []
        for item in ds:
            out.append({
                '_id': item['id'],
                'question': item['question'],
                'answer': item['answer'],
                'type': item['type'],
                'level': item['level'],
                'context': list(zip(item['context']['title'], item['context']['sentences'])),
                'supporting_facts': list(zip(item['supporting_facts']['title'], item['supporting_facts']['sent_id']))
            })
        print('Loaded HotpotQA from Hugging Face.')
        return out
    except Exception as e:
        print(f'Hugging Face load failed ({e}). Trying local/official file...')

    if os.path.exists(DATA_PATH) and os.path.getsize(DATA_PATH) > 100000:
        return json.load(open(DATA_PATH, encoding='utf-8'))
        
    url = 'http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json'
    urllib.request.urlretrieve(url, DATA_PATH)
    return json.load(open(DATA_PATH, encoding='utf-8'))

data = load_hotpot_dev()
print(f'Loaded {len(data)} total dev questions.')

with open(DATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(data, f)

rng = random.Random(SEED)
pool_qs = rng.sample(data, min(N_CORPUS, len(data)))
eval_qs = pool_qs[:min(N_EVAL, len(pool_qs))]

corpus = {}
for q in pool_qs:
    for title, sents in q['context']:
        corpus.setdefault(title, ' '.join(s.strip() for s in sents))
pids = list(corpus.keys())

for q in eval_qs:
    q['gold_pids'] = sorted({t for t, _ in q['supporting_facts']})
    assert all(g in corpus for g in q['gold_pids'])

print(f'Pooled corpus size: {len(pids)} unique passages.')
print(f'Evaluation subset: {len(eval_qs)} questions.')
print('Question breakdown:', collections.Counter(q['type'] for q in eval_qs))

with open(os.path.join(OUT, 'eval_qs.json'), 'w') as f:
    json.dump(eval_qs, f)
with open(os.path.join(OUT, 'corpus.json'), 'w') as f:
    json.dump(corpus, f)

## Step 4: Metric Definitions & Unit Tests

In [ ]:
def normalize_answer(s):
    s = str(s).lower()
    s = ''.join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return ' '.join(s.split())

def em(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def f1(pred, gold):
    np_, ng = normalize_answer(pred), normalize_answer(gold)
    if np_ in ('yes', 'no', 'noanswer') and np_ != ng: return 0.0
    if ng in ('yes', 'no', 'noanswer') and np_ != ng: return 0.0
    p, g = np_.split(), ng.split()
    same = sum((collections.Counter(p) & collections.Counter(g)).values())
    if same == 0: return 0.0
    pr, rc = same / len(p), same / len(g)
    return 2 * pr * rc / (pr + rc)

def gold_ranks(ranked, gold):
    return {g: (ranked.index(g) + 1 if g in ranked else None) for g in gold}

def sf_recall(ranked, gold, k=TOP_K):
    return len(set(ranked[:k]) & set(gold)) / max(len(gold), 1)

def all_sf(ranked, gold, k=TOP_K):
    return float(set(gold) <= set(ranked[:k]))

def mrr(ranked, gold):
    r = [i + 1 for i, p in enumerate(ranked) if p in gold]
    return 1.0 / r[0] if r else 0.0

def paired_bootstrap(a, b, B=10000, seed=SEED):
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    r = np.random.default_rng(seed)
    n = len(d)
    if n == 0:
        return 0.0, 0.0, 0.0, 1.0
    boot_indices = r.integers(0, n, (B, n))
    means = d[boot_indices].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    p = min(1.0, 2.0 * min(np.mean(means <= 0), np.mean(means >= 0)))
    return float(d.mean()), float(lo), float(hi), float(max(p, 1.0 / B))

def holm_bonferroni(pvals):
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.zeros(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        running_max = max(running_max, (m - rank) * pvals[idx])
        adj[idx] = min(1.0, running_max)
    return adj.tolist()

# Unit Tests
assert em('The Eiffel Tower', 'eiffel tower') == 1.0
assert em('Paris', 'London') == 0.0
assert abs(f1('New York City', 'New York') - 0.8) < 1e-9
assert f1('yes', 'no') == 0.0 and f1('yes', 'yes') == 1.0
print('All metric and statistical unit tests passed successfully.')

## Step 5: Robust Tracker & Context-Hash Cache

In [ ]:
class Tracker:
    def __init__(self, path=os.path.join(OUT, 'tracker.csv')):
        self.path = path
        self.rows = []
        if os.path.exists(path):
            try:
                self.rows = pd.read_csv(path).to_dict('records')
                print(f'Loaded {len(self.rows)} existing tracking records.')
            except Exception:
                self.rows = []

    def log(self, qid, qtype, stage, gold, ranked, k=TOP_K, pred=None, answer=None, tag=''):
        row = dict(qid=qid, type=qtype, stage=stage, gold_pids=json.dumps(gold),
                   retrieved_pids=json.dumps(ranked[:50]),
                   gold_ranks=json.dumps(gold_ranks(ranked, gold)),
                   sf_recall=sf_recall(ranked, gold, k), all_sf=all_sf(ranked, gold, k),
                   mrr=mrr(ranked, gold), prediction=pred, em=None, f1=None, error_tag=tag)
        if pred is not None and answer is not None:
            row['em'] = em(pred, answer)
            row['f1'] = f1(pred, answer)
        self.rows = [r for r in self.rows if not (str(r['qid']) == str(qid) and r['stage'] == stage)]
        self.rows.append(row)

    def save(self):
        pd.DataFrame(self.rows).to_csv(self.path, index=False)

    def df(self, stage=None):
        df_all = pd.DataFrame(self.rows)
        if df_all.empty: return df_all
        if stage:
            return df_all[df_all['stage'] == stage].set_index('qid').sort_index()
        return df_all

class AnswerCache:
    def __init__(self, path=os.path.join(CACHE_DIR, 'answers.pkl')):
        self.path = path
        self.d = pickle.load(open(path, 'rb')) if os.path.exists(path) else {}
        
    def _key(self, system, qid, context_pids):
        ctx_sig = hashlib.md5('::'.join(context_pids).encode('utf-8')).hexdigest()[:8]
        return (system, str(qid), ctx_sig)
        
    def get(self, system, qid, context_pids):
        return self.d.get(self._key(system, qid, context_pids))
        
    def put(self, system, qid, context_pids, text):
        self.d[self._key(system, qid, context_pids)] = text
        
    def save(self):
        pickle.dump(self.d, open(self.path, 'wb'))

tracker = Tracker()
answers = AnswerCache()
print('Tracker & Context-Hash Cache initialized.')

## Step 6: Retrieval & Reranker Modules (Memory-Safe for T4 GPU)

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device: {device}')

def tokenize(text):
    return re.findall(r'\w+', text.lower())

# 1. BM25
bm25_cache = os.path.join(CACHE_DIR, 'bm25.pkl')
if os.path.exists(bm25_cache):
    with open(bm25_cache, 'rb') as f:
        bm25_index = pickle.load(f)
    print('[Cache] Loaded BM25 index.')
else:
    tokenized_corpus = [tokenize(f"{pid} {corpus[pid]}") for pid in pids]
    bm25_index = BM25Okapi(tokenized_corpus)
    with open(bm25_cache, 'wb') as f:
        pickle.dump(bm25_index, f)

def retrieve_bm25(query, k=50):
    tokens = tokenize(query)
    scores = bm25_index.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return [pids[i] for i in top_idx]

# 2. Dense Embeddings (BAAI/bge-small-en-v1.5)
embedder = SentenceTransformer('BAAI/bge-small-en-v1.5', device=device)
dense_cache = os.path.join(CACHE_DIR, 'dense_embeddings.npy')
if os.path.exists(dense_cache):
    corpus_embeddings = np.load(dense_cache)
    print('[Cache] Loaded Dense embeddings.')
else:
    passages = [f"{pid}: {corpus[pid]}" for pid in pids]
    corpus_embeddings = embedder.encode(passages, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save(dense_cache, corpus_embeddings)

BGE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

def retrieve_dense(query, k=50):
    formatted_q = BGE_QUERY_PREFIX + query
    with torch.no_grad():
        q_emb = embedder.encode(formatted_q, normalize_embeddings=True, show_progress_bar=False)
    sims = np.dot(corpus_embeddings, q_emb)
    top_idx = np.argsort(sims)[::-1][:k]
    return [pids[i] for i in top_idx]

# 3. Reciprocal Rank Fusion (RRF)
def rrf_merge(rankings_list, k_rrf=60):
    scores = collections.defaultdict(float)
    for ranked in rankings_list:
        for rank, pid in enumerate(ranked):
            scores[pid] += 1.0 / (k_rrf + rank + 1)
    return sorted(scores.keys(), key=lambda x: scores[x], reverse=True)

def retrieve_hybrid(query, k=50):
    bm25_res = retrieve_bm25(query, k=100)
    dense_res = retrieve_dense(query, k=100)
    return rrf_merge([bm25_res, dense_res])[:k]

# 4. Cross-Encoder Reranker (max_length=256 for fast 4.5s speed and low VRAM)
print('Loading cross-encoder reranker...')
reranker = CrossEncoder('BAAI/bge-reranker-base', device=device, max_length=256)

def rerank(query, candidate_pids, top_k=TOP_K):
    if not candidate_pids: return []
    pairs = [[query, f"{pid}: {corpus[pid]}"] for pid in candidate_pids]
    with torch.no_grad():
        scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)
    ranked = [pid for _, pid in sorted(zip(scores, candidate_pids), key=lambda x: x[0], reverse=True)]
    return ranked[:top_k]

print('All retrieval modules loaded successfully!')

## Step 7: Generator Setup (Qwen-2.5-3B-Instruct on GPU)

In [ ]:
from transformers import pipeline
print('Loading Qwen/Qwen2.5-3B-Instruct onto GPU...')
llm_pipeline = pipeline('text-generation', model='Qwen/Qwen2.5-3B-Instruct',
                        device_map='auto', torch_dtype=torch.float16)

def call_llm(prompt):
    messages = [{'role': 'user', 'content': prompt}]
    out = llm_pipeline(messages, max_new_tokens=32, do_sample=False)
    return out[0]['generated_text'][-1]['content'].strip()

def generate_answer(system_name, qid, query, context_pids):
    # Check context-hash protected cache
    cached_ans = answers.get(system_name, qid, context_pids)
    if cached_ans is not None:
        return cached_ans

    # Dedicated Closed-Book prompt for S0
    if not context_pids:
        prompt = (f'Answer the following question directly and concisely with just the entity name, '
                  f'number, or yes/no.\n\nQuestion: {query}\nAnswer:')
    else:
        context_str = '\n\n'.join(f'Document: {p}\n{corpus[p]}' for p in context_pids if p in corpus)
        prompt = (f'Answer the question concisely based only on the provided documents. '
                  f'Answer with just the entity name, number, or yes/no.\n\n'
                  f'Documents:\n{context_str}\n\nQuestion: {query}\nAnswer:')

    pred = None
    for attempt in range(4):
        try:
            pred = call_llm(prompt)
            if pred:
                break
        except Exception as e:
            time.sleep(2 ** attempt)

    if pred is None:
        pred = 'unknown'

    answers.put(system_name, qid, context_pids, pred)
    return pred

print('Generator active: Qwen-2.5-3B-Instruct (GPU)')

## Step 8: Run Systems ($S0$–$S9$) & Controls

In [ ]:
print(f'Running Systems on {len(eval_qs)} questions...')

for i, q in enumerate(tqdm(eval_qs, desc='Evaluating Systems')):
    qid, query, gold, ans, qtype = q['_id'], q['question'], q['gold_pids'], q['answer'], q['type']

    # S0: Closed Book
    s0_pred = generate_answer('S0_ClosedBook', qid, query, [])
    tracker.log(qid, qtype, 'S0_ClosedBook', gold, [], k=TOP_K, pred=s0_pred, answer=ans)

    # S1: BM25
    s1_ranked = retrieve_bm25(query, k=TOP_K)
    s1_pred = generate_answer('S1_BM25', qid, query, s1_ranked)
    tracker.log(qid, qtype, 'S1_BM25', gold, s1_ranked, k=TOP_K, pred=s1_pred, answer=ans)

    # S2: Dense
    s2_ranked = retrieve_dense(query, k=TOP_K)
    s2_pred = generate_answer('S2_Dense', qid, query, s2_ranked)
    tracker.log(qid, qtype, 'S2_Dense', gold, s2_ranked, k=TOP_K, pred=s2_pred, answer=ans)

    # S3: Hybrid RRF
    s3_ranked = retrieve_hybrid(query, k=TOP_K)
    s3_pred = generate_answer('S3_Hybrid', qid, query, s3_ranked)
    tracker.log(qid, qtype, 'S3_Hybrid', gold, s3_ranked, k=TOP_K, pred=s3_pred, answer=ans)

    # S5: Hybrid + Reranker (Pool=25)
    s5_pool25 = retrieve_hybrid(query, k=25)
    s5_ranked = rerank(query, s5_pool25, top_k=TOP_K)
    s5_pred = generate_answer('S5_Hybrid_Rerank', qid, query, s5_ranked)
    tracker.log(qid, qtype, 'S5_Hybrid_Rerank', gold, s5_ranked, k=TOP_K, pred=s5_pred, answer=ans)

    # S5_Ctrl_Pool50: Hybrid + Reranker with Pool=50 (Fair candidate pool control vs S8)
    s5_pool50 = retrieve_hybrid(query, k=50)
    s5_50_ranked = rerank(query, s5_pool50, top_k=TOP_K)
    s5_50_pred = generate_answer('S5_Ctrl_Pool50', qid, query, s5_50_ranked)
    tracker.log(qid, qtype, 'S5_Ctrl_Pool50', gold, s5_50_ranked, k=TOP_K, pred=s5_50_pred, answer=ans)

    # Ctrl_SinglePass_K10: Single-pass Hybrid with K=10 passages (Context budget control)
    ctrl_k10_ranked = retrieve_hybrid(query, k=10)
    ctrl_k10_pred = generate_answer('Ctrl_SinglePass_K10', qid, query, ctrl_k10_ranked)
    tracker.log(qid, qtype, 'Ctrl_SinglePass_K10', gold, ctrl_k10_ranked, k=10, pred=ctrl_k10_pred, answer=ans)

    # S7: TRUE Iterative Query Augmentation Baseline (Dense-only 2-hop)
    hop1_dense = retrieve_dense(query, k=2)
    hop1_snippet = ' '.join(corpus.get(p, '') for p in hop1_dense)
    hop2_dense = retrieve_dense(f"{query} {hop1_snippet[:200]}", k=10)
    hop2_candidates = [p for p in hop2_dense if p not in hop1_dense][:3]
    s7_ranked = (hop1_dense + hop2_candidates)[:TOP_K]
    s7_pred = generate_answer('S7_Iterative_Dense', qid, query, s7_ranked)
    tracker.log(qid, qtype, 'S7_Iterative_Dense', gold, s7_ranked, k=TOP_K, pred=s7_pred, answer=ans)

    # S8: Full Multi-Hop Pipeline (Hybrid + Reranker + Evidence-Conditioned Hop 2)
    h1_pool = retrieve_hybrid(query, k=25)
    h1_top = rerank(query, h1_pool, top_k=3)
    h1_evidence = ' '.join(corpus.get(p, '') for p in h1_top[:2])
    h2_pool = retrieve_hybrid(f"{query} {h1_evidence[:200]}", k=25)
    merged_candidates = list(collections.OrderedDict.fromkeys(h1_pool + h2_pool))
    s8_ranked = rerank(query, merged_candidates, top_k=TOP_K)
    s8_pred = generate_answer('S8_Full_Pipeline', qid, query, s8_ranked)

    # Diagnostic Tag: Did gold appear in Hop 1 pool, Hop 2 pool, or missed entirely?
    gold_set = set(gold)
    if gold_set <= set(h1_pool):
        recovery_tag = 'Hop1_Pool'
    elif gold_set <= set(merged_candidates):
        recovery_tag = 'Hop2_Added'
    else:
        recovery_tag = 'Never_Retrieved'
    tracker.log(qid, qtype, 'S8_Full_Pipeline', gold, s8_ranked, k=TOP_K, pred=s8_pred, answer=ans, tag=recovery_tag)

    # S9: Oracle
    s9_ranked = gold[:TOP_K]
    s9_pred = generate_answer('S9_Oracle', qid, query, s9_ranked)
    tracker.log(qid, qtype, 'S9_Oracle', gold, s9_ranked, k=TOP_K, pred=s9_pred, answer=ans)

    if (i + 1) % 25 == 0:
        tracker.save()
        answers.save()

tracker.save()
answers.save()
print('Systems S0-S9 completed and saved.')

## Step 9: Run Leave-One-Out Ablation Study on $S8$

In [ ]:
print('Running Ablation Study on S8...')

for i, q in enumerate(tqdm(eval_qs, desc='Evaluating Ablations')):
    qid, query, gold, ans, qtype = q['_id'], q['question'], q['gold_pids'], q['answer'], q['type']

    # Ablation: S8 - Reranker
    h1_pool = retrieve_hybrid(query, k=25)
    h1_ev = ' '.join(corpus.get(p, '') for p in h1_pool[:2])
    h2_pool = retrieve_hybrid(f"{query} {h1_ev[:200]}", k=25)
    a_norerank_ranked = rrf_merge([h1_pool, h2_pool])[:TOP_K]
    a_norerank_pred = generate_answer('Abl_No_Reranker', qid, query, a_norerank_ranked)
    tracker.log(qid, qtype, 'Abl_No_Reranker', gold, a_norerank_ranked, k=TOP_K, pred=a_norerank_pred, answer=ans)

    # Ablation: S8 - BM25 (Dense-only)
    h1_dense = retrieve_dense(query, k=25)
    h1_top = rerank(query, h1_dense, top_k=3)
    h1_ev = ' '.join(corpus.get(p, '') for p in h1_top[:2])
    h2_dense = retrieve_dense(f"{query} {h1_ev[:200]}", k=25)
    a_dense_ranked = rerank(query, list(collections.OrderedDict.fromkeys(h1_dense + h2_dense)), top_k=TOP_K)
    a_dense_pred = generate_answer('Abl_Dense_Only', qid, query, a_dense_ranked)
    tracker.log(qid, qtype, 'Abl_Dense_Only', gold, a_dense_ranked, k=TOP_K, pred=a_dense_pred, answer=ans)

    # Ablation: S8 - Dense (BM25-only)
    h1_bm25 = retrieve_bm25(query, k=25)
    h1_top = rerank(query, h1_bm25, top_k=3)
    h1_ev = ' '.join(corpus.get(p, '') for p in h1_top[:2])
    h2_bm25 = retrieve_bm25(f"{query} {h1_ev[:200]}", k=25)
    a_bm25_ranked = rerank(query, list(collections.OrderedDict.fromkeys(h1_bm25 + h2_bm25)), top_k=TOP_K)
    a_bm25_pred = generate_answer('Abl_BM25_Only', qid, query, a_bm25_ranked)
    tracker.log(qid, qtype, 'Abl_BM25_Only', gold, a_bm25_ranked, k=TOP_K, pred=a_bm25_pred, answer=ans)

    if (i + 1) % 25 == 0:
        tracker.save()
        answers.save()

tracker.save()
answers.save()
print('Ablation study completed and saved.')

## Step 10: Publication-Ready Tables & Multi-Metric Statistical Testing

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

tracker.save()
df = tracker.df()

stages_order = [
    'S0_ClosedBook', 'S1_BM25', 'S2_Dense', 'S3_Hybrid',
    'S5_Hybrid_Rerank', 'S5_Ctrl_Pool50', 'Ctrl_SinglePass_K10',
    'S7_Iterative_Dense', 'S8_Full_Pipeline', 'S9_Oracle',
    'Abl_No_Reranker', 'Abl_Dense_Only', 'Abl_BM25_Only'
]

def summarize_df(data_df, title='Summary'):
    rows = []
    for s in stages_order:
        sdf = data_df[data_df.stage == s]
        if len(sdf) == 0: continue
        rows.append({
            'System': s,
            'N': len(sdf),
            'AllSF@5 (%)': sdf['all_sf'].mean() * 100,
            'SF_Recall@5 (%)': sdf['sf_recall'].mean() * 100,
            'MRR': sdf['mrr'].mean(),
            'EM (%)': sdf['em'].mean() * 100 if sdf['em'].notnull().all() else np.nan,
            'F1 (%)': sdf['f1'].mean() * 100 if sdf['f1'].notnull().all() else np.nan
        })
    res = pd.DataFrame(rows).round(2)
    print(f'\n=== {title} ===')
    display(res)
    return res

# 1. Main Overall Results Table
overall_table = summarize_df(df, f'OVERALL PERFORMANCE (N={len(df.qid.unique())})')
overall_table.to_csv(os.path.join(OUT, 'results_overall.csv'), index=False)

# 2. Question-Type Subgroup Analysis
bridge_table = summarize_df(df[df.type == 'bridge'], 'BRIDGE QUESTIONS')
comp_table = summarize_df(df[df.type == 'comparison'], 'COMPARISON QUESTIONS')

# 3. Statistical Significance Testing
print('\n' + '='*70)
print('PAIRED BOOTSTRAP SIGNIFICANCE TESTING (B=10,000, 95% CI, Holm-Corrected)')
print('='*70)

tested_baselines = [
    'S2_Dense',          # Single-pass dense baseline
    'S3_Hybrid',         # Strongest single-pass baseline
    'S5_Hybrid_Rerank',  # Single-pass + reranker (pool=25)
    'S5_Ctrl_Pool50',    # Single-pass + reranker (pool=50: fair pool control)
    'S7_Iterative_Dense',# Published-style iterative baseline
    'Abl_No_Reranker',   # Component ablation: no reranker
    'Abl_Dense_Only',    # Component ablation: dense only
    'Abl_BM25_Only'      # Component ablation: BM25 only
]

metrics_to_test = ['all_sf', 'f1', 'em']
s8_data = tracker.df('S8_Full_Pipeline')

for metric in metrics_to_test:
    stat_rows = []
    p_vals = []
    for b in tested_baselines:
        b_data = tracker.df(b)
        common_ids = s8_data.index.intersection(b_data.index)
        if len(common_ids) == 0: continue
        mean_diff, lo, hi, pval = paired_bootstrap(s8_data.loc[common_ids, metric],
                                                   b_data.loc[common_ids, metric])
        stat_rows.append({
            'Comparison': f'S8 vs {b}',
            'Metric': metric.upper(),
            'Mean Diff (%)': mean_diff * 100,
            '95% CI (%)': f'[{lo*100:.2f}, {hi*100:.2f}]',
            'Raw p-val': pval
        })
        p_vals.append(pval)

    if p_vals:
        adj_p = holm_bonferroni(p_vals)
        for r, p_adj in zip(stat_rows, adj_p):
            r['Holm p'] = f'{p_adj:.4f}'
            r['Significant (alpha=0.05)'] = 'Yes' if p_adj < 0.05 and r['Mean Diff (%)'] > 0 else 'No'

        stat_df = pd.DataFrame(stat_rows)
        print(f'\n--- Statistical Significance on {metric.upper()} ---')
        display(stat_df)
        stat_df.to_csv(os.path.join(OUT, f'significance_{metric}.csv'), index=False)

# 4. Supporting Fact Recovery Figure
s8_records = tracker.df('S8_Full_Pipeline')
if 'error_tag' in s8_records.columns and not s8_records['error_tag'].isna().all():
    counts = s8_records['error_tag'].value_counts()
    plt.figure(figsize=(7, 4))
    palette = {'Hop1_Pool': '#2ecc71', 'Hop2_Added': '#3498db', 'Never_Retrieved': '#e74c3c'}
    sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette=palette, legend=False)
    plt.title('Stage of Complete Supporting Fact Recovery in S8 Pool')
    plt.ylabel('Question Count')
    plt.xlabel('Recovery Stage')
    for idx, val in enumerate(counts.values):
        plt.text(idx, val + 0.5, f'{val} ({val/len(s8_records)*100:.1f}%)', ha='center', fontweight='bold')
    plt.tight_layout()
    fig_path = os.path.join(OUT, 'recovery_stages_figure.png')
    plt.savefig(fig_path, dpi=300)
    plt.show()
    print(f'Diagnostic figure saved to {fig_path}')

# 5. Multi-Retriever Depth Curve
print('\n' + '='*70)
print('MULTI-RETRIEVER DEPTH CURVE (AllSF @ K=5, 10, 25, 50)')
print('='*70)
Ks = [5, 10, 25, 50]
fns = {'BM25': retrieve_bm25, 'Dense': retrieve_dense, 'Hybrid': retrieve_hybrid}
depth_rows = []
for q in eval_qs:
    for name, fn in fns.items():
        r = fn(q['question'], k=50)
        depth_rows.append({'type': q['type'], 'retriever': name,
                           **{f'AllSF@{k}': all_sf(r, q['gold_pids'], k) for k in Ks}})
depth_df = pd.DataFrame(depth_rows)
display((depth_df.groupby(['retriever', 'type']).mean(numeric_only=True) * 100).round(1))
depth_df.to_csv(os.path.join(OUT, 'depth_curve.csv'), index=False)
print('Pipeline complete. All outputs saved to Google Drive / local directory.')